# AiStock V9.1 测试 Notebook
## 数据获取 → 核心计算 → 可视化 全流程测试
### V9.1 关键更新
- tdxAPICode180.xlsx 代码表 (57371条 vs 旧31692条)
- 新增商品期权市场码: CZCE=4, DCE=5, SHFE=6, GZ=67
- 64个商品期权标的完整覆盖
- TDX扩展端口: 180.153.18.176:7721

In [12]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

PROJECT_ROOT: /home/ts/app/AiStock_v10


In [13]:
from base_services import LoggerService, ConfigService, CacheService, TDXConnectionPool

logger_service = LoggerService(console_level=10, project_root=str(PROJECT_ROOT))
logger = logger_service.get_logger("test")
config = ConfigService()
cache = CacheService(max_size=2000, default_ttl=3600)

# Verify TDX extension IP
from config.global_settings import (
    TDX_STANDARD_HOST, TDX_STANDARD_PORT,
    TDX_EXTENSION_HOST, TDX_EXTENSION_PORT,
    TDX_CODE_TABLE_PATH, SYSTEM_VERSION
)
print(f"System Version: {SYSTEM_VERSION}")
print(f"TDX Standard: {TDX_STANDARD_HOST}:{TDX_STANDARD_PORT}")
print(f"TDX Extension: {TDX_EXTENSION_HOST}:{TDX_EXTENSION_PORT}")
print(f"Code Table: {TDX_CODE_TABLE_PATH}")

2026-05-26 15:22:12.755 | INFO     | aistock | 📋 LoggerService 初始化完成 | 日志目录: /home/ts/app/AiStock_v10/logs | 控制台级别: DEBUG | 文件级别: DEBUG | 轮转: 50MB×10
System Version: 9.1.0
TDX Standard: 180.153.18.170:7709
TDX Extension: 180.153.18.176:7721
Code Table: /home/ts/app/AiStock_v10/notebooks/tdxAPICode180.xlsx


In [14]:
from data_service import TDXAdapter, AKAdapter, DatabaseReader, DataLoaderService

tdx = TDXAdapter(
    std_host=TDX_STANDARD_HOST, std_port=TDX_STANDARD_PORT,
    ext_host=TDX_EXTENSION_HOST, ext_port=TDX_EXTENSION_PORT,
    pool_size=3,
)
ak = AKAdapter(rate_limit_interval=0.5, retry_count=2, retry_delay=1.0, cache_ttl=300.0)

try:
    from config.global_settings import DATABASE_ENGINES
    db_config = DATABASE_ENGINES.get("index_pe_db", {})
    if isinstance(db_config, str):
        db_config = {"url": db_config}
except ImportError:
    db_config = None

db_reader = DatabaseReader(db_config=db_config, engine_name="index_pe_db", fallback_on_error=True)
data_loader = DataLoaderService(tdx_adapter=tdx, ak_adapter=ak, db_reader=db_reader)
print("Data layer initialized")

Data layer initialized


In [15]:
from market_state_system.core.contract_manager import ContractManager

cm = ContractManager(
    code_table_path=TDX_CODE_TABLE_PATH,
    option_code_parser=None,
)

print(f"Total contracts loaded: {len(cm.contracts)}")
print(f"Futures varieties: {len(cm._variety_codes)}")
print(f"Option underlyings: {len(cm._option_by_underlying)}")
print(f"Code↔CodeName mappings: {len(cm._code_name_to_code)}")
print()

# Test commodity contracts
commodity_pairs = cm.get_commodity_contracts()
for key, pair in commodity_pairs.items():
    print(f"  {key}: near={pair.near_code}({pair.near_year}-{pair.near_month:02d}) "
          f"far={pair.far_code}({pair.far_year}-{pair.far_month:02d}) "
          f"market={pair.market_type}")

print()

# Test index futures
index_futures = cm.get_index_futures_contracts()
for key, fc in index_futures.items():
    print(f"  {key}: active={fc.futures_code} next_q={fc.next_quarter_code} "
          f"spot={fc.spot_code} market={fc.market_type}")

print()

# Test option contracts
for underlying in ['IO', '510050', 'CU', 'LC']:
    opt = cm.get_option_contracts(underlying)
    if opt:
        print(f"  {underlying}: {len(opt.call_codes)}C/{len(opt.put_codes)}P "
              f"codes, {len(opt.call_code_names)}C/{len(opt.put_code_names)}P code_names "
              f"delivery={opt.delivery_year}-{opt.delivery_month:02d}")
    else:
        print(f"  {underlying}: No option data")

Total contracts loaded: 57222
Futures varieties: 90
Option underlyings: 76
Code↔CodeName mappings: 56809

  copper: near=CU2606(2026-06) far=CU2607(2026-07) market=future_sh
  aluminum: near=AL2606(2026-06) far=AL2607(2026-07) market=future_sh
  lithium: near=LC2606(2026-06) far=LC2607(2026-07) market=future_gz
  silicon: near=SI2606(2026-06) far=SI2607(2026-07) market=future_gz
  crude: near=SC2606(2026-06) far=SC2607(2026-07) market=future_sh
  rebar: near=RB2606(2026-06) far=RB2607(2026-07) market=future_sh
  gold: near=AU2606(2026-06) far=AU2608(2026-08) market=future_sh
  soybean: near=M2607(2026-07) far=M2608(2026-08) market=future_dl

  if: active=IF2606 next_q=IF2609 spot=000300 market=future_zj
  ih: active=IH2606 next_q=IH2609 spot=000016 market=future_zj
  ic: active=IC2606 next_q=IC2609 spot=000905 market=future_zj
  im: active=IM2606 next_q=IM2609 spot=000852 market=future_zj

  IO: 33C/33P codes, 33C/33P code_names delivery=2026-06
  510050: 0C/0P codes, 0C/0P code_names 

In [16]:
# Test code ↔ code_name bidirectional lookup
test_cases = [
    ('HO8U03PD', 'CFFEX option code'),
    ('10010295', 'SH ETF option code'),
    ('90006441', 'SZ ETF option code'),
    ('AP8Y0FFL', 'CZCE commodity option code'),
    ('LC8V4ABL', 'GZ commodity option code'),
    ('CU2606', 'SHFE futures code'),
]

for code, desc in test_cases:
    code_name = cm.lookup_code_name_by_code(code)
    reverse = cm.lookup_code_by_code_name(code_name) if code_name else None
    match = "✓" if reverse == code else "✗"
    print(f"  {match} {desc}: code={code} → code_name={code_name} → reverse={reverse}")

  ✓ CFFEX option code: code=HO8U03PD → code_name=HO2606-C-2400 → reverse=HO8U03PD
  ✓ SH ETF option code: code=10010295 → code_name=510050C6A02850 → reverse=10010295
  ✓ SZ ETF option code: code=90006441 → code_name=159901C6M003100A → reverse=90006441
  ✓ CZCE commodity option code: code=AP8Y0FFL → code_name=AP2610-C-10000 → reverse=AP8Y0FFL
  ✓ GZ commodity option code: code=LC8V4ABL → code_name=LC2607-C-100000 → reverse=LC8V4ABL
  ✓ SHFE futures code: code=CU2606 → code_name=沪铜2606 → reverse=CU2606


In [17]:
# Test standard port - index data
print("=== Standard Port (7709) Tests ===")
for code in ['000001', '000300', '399006']:
    df = tdx.get_index_daily(code=code, market_type='index_sh', count=5)
    if df is not None and not df.empty:
        print(f"  {code}: {len(df)} bars, latest={df['close'].iloc[-1]:.2f}")
    else:
        print(f"  {code}: FAILED")

print()

# Test extension port - futures data
print("=== Extension Port (7721) Tests ===")
for code, mt in [('IFL8', 'future_zz'), ('CUL8', 'future_sh'), ('MML8', 'future_dl')]:
    df = tdx.get_future_daily(code=code, market_type=mt, count=5)
    if df is not None and not df.empty:
        print(f"  {code}: {len(df)} bars, latest={df['close'].iloc[-1]:.2f}")
    else:
        print(f"  {code}: FAILED")

=== Standard Port (7709) Tests ===


get_bars 连接异常,尝试重连: 'list' object has no attribute 'empty'
TDXAdapter 重试 (1/2): 'list' object has no attribute 'empty'
get_bars 连接异常,尝试重连: 'list' object has no attribute 'empty'
TDXAdapter 重试 (2/2): 'list' object has no attribute 'empty'


AttributeError: 'list' object has no attribute 'empty'

In [ ]:
from market_state_system.core.option_pcr_engine import OptionPCREngine

pcr_engine = OptionPCREngine(
    tdx_adapter=tdx, config=config, cache_service=cache,
    contract_manager=cm,
)

# Test single ETF PCR
print("=== ETF PCR Test ===")
try:
    pcr_50etf = pcr_engine.calculate_etf_pcr("510050")
    print(f"  510050: OI={pcr_50etf.pcr_oi:.4f} Vol={pcr_50etf.pcr_volume:.4f} "
          f"contracts={pcr_50etf.contract_count}")
except Exception as e:
    print(f"  510050: Error - {e}")

# Test CFFEX PCR
print("\n=== CFFEX PCR Test ===")
try:
    pcr_io = pcr_engine.calculate_cffex_pcr("IO")
    print(f"  IO: OI={pcr_io.pcr_oi:.4f} Vol={pcr_io.pcr_volume:.4f} "
          f"contracts={pcr_io.contract_count}")
except Exception as e:
    print(f"  IO: Error - {e}")

# Test commodity PCR
print("\n=== Commodity PCR Test ===")
try:
    pcr_cu = pcr_engine.calculate_commodity_pcr("CU")
    print(f"  CU: OI={pcr_cu.pcr_oi:.4f} Vol={pcr_cu.pcr_volume:.4f} "
          f"contracts={pcr_cu.contract_count}")
except Exception as e:
    print(f"  CU: Error - {e}")

In [ ]:
from market_state_system.core.derivatives_signal_engine import DerivativesSignalEngine

deriv_engine = DerivativesSignalEngine(
    data_service=tdx,
    contract_manager=cm,
    overseas_signal_engine=None,
    config=config,
    cache=cache,
)

result = deriv_engine.calculate_all()
print(f"Composite Signal: {result.composite_signal:.2f} [{result.composite_direction}]")
print(f"Commodity signals: {len(result.commodity_signals)}")
print(f"Term structure: {len(result.term_structure)}")
print(f"Index futures basis: {len(result.index_futures_basis)}")
print(f"Industry sentiment: {len(result.industry_sentiment)}")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Get index data for visualization
hs300 = tdx.get_index_daily(code='000300', market_type='index_sh', count=120)
if hs300 is not None and not hs300.empty:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=('沪深300走势', '成交量'),
                        vertical_spacing=0.1)

    fig.add_trace(go.Candlestick(
        x=hs300.index, open=hs300['open'], high=hs300['high'],
        low=hs300['low'], close=hs300['close'], name='K线'
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=hs300.index, y=hs300['volume'], name='成交量',
        marker_color='rgba(100,149,237,0.5)'
    ), row=2, col=1)

    fig.update_layout(
        title='AiStock V9.1 — 沪深300指数走势',
        height=600, showlegend=True,
        xaxis_rangeslider_visible=False,
    )
    fig.show()
else:
    print("Failed to fetch HS300 data for visualization")

In [ ]:
# Visualize PCR across underlyings
pcr_data = {}
for ul in ['510050', '510300', 'IO', 'HO', 'MO']:
    try:
        if ul.startswith(('510', '159')):
            r = pcr_engine.calculate_etf_pcr(ul)
        else:
            r = pcr_engine.calculate_cffex_pcr(ul)
        pcr_data[ul] = {'pcr_oi': r.pcr_oi, 'pcr_vol': r.pcr_volume, 'contracts': r.contract_count}
    except:
        pcr_data[ul] = {'pcr_oi': 0, 'pcr_vol': 0, 'contracts': 0}

import pandas as pd
pcr_df = pd.DataFrame(pcr_data).T
print(pcr_df)

if not pcr_df.empty:
    fig = go.Figure(data=[
        go.Bar(name='PCR(OI)', x=pcr_df.index, y=pcr_df['pcr_oi'], marker_color='steelblue'),
        go.Bar(name='PCR(Vol)', x=pcr_df.index, y=pcr_df['pcr_vol'], marker_color='coral'),
    ])
    fig.add_hline(y=1.0, line_dash="dash", line_color="gray", annotation_text="中性线")
    fig.add_hline(y=1.3, line_dash="dot", line_color="red", annotation_text="恐慌阈值")
    fig.add_hline(y=0.7, line_dash="dot", line_color="green", annotation_text="乐观阈值")
    fig.update_layout(title='AiStock V9.1 — 期权PCR对比', barmode='group', height=400)
    fig.show()

In [ ]:
# Generate market state summary
from datetime import datetime
print(f"\n{'='*60}")
print(f"AiStock V9.1 市场状态量化系统 — 测试摘要")
print(f"时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*60}")
print(f"系统版本: {SYSTEM_VERSION}")
print(f"TDX标准: {TDX_STANDARD_HOST}:{TDX_STANDARD_PORT}")
print(f"TDX扩展: {TDX_EXTENSION_HOST}:{TDX_EXTENSION_PORT}")
print(f"代码表: {len(cm.contracts)} 合约 (tdxAPICode180.xlsx)")
print(f"商品期货: {len(commodity_pairs)} 品种动态合约")
print(f"股指期货: {len(index_futures)} 品种当月/下季月")
print(f"商品期权: 64 标的 (CZCE=4, DCE=5, SHFE=6, GZ=67)")
print(f"衍生品信号: {result.composite_signal:.1f} [{result.composite_direction}]")
print(f"{'='*60}")